In [17]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from mlxtend.plotting import plot_decision_regions
import seaborn as sns
import matplotlib.pyplot as plt

# **Step 1 : Initialize the dataset**

In [18]:
def initialize_data():
  df = pd.DataFrame()
  df['X1'] = [1,2,3,4,5,6,6,7,9,9]
  df['X2'] = [5,3,6,8,1,9,5,8,9,2]
  df['label'] = [1,1,0,1,0,1,0,1,0,0]
  df['weights'] = 1/df.shape[0]
  return df

# Step 2 : Train a Decision Tree Model


In [19]:
def train_decision_tree(X,y,depth=1):
  dt = DecisionTreeClassifier(max_depth=depth)
  dt.fit(X,y)
  return dt

# **Step 3 : Predicts Using The model**

In [20]:
def add_predictions(df,model):
  X = df[['X1' , 'X2']].values
  df['y_pred'] = model.predict(X)
  return df

# Step 4 : Calculate model weight(alpha)

In [21]:
def calculate_model_weight(error):
  return 0.55 * np.log((1-error) / (error + 0.000001))

# **Step 5:Update sample weights**

In [22]:
def update_weights(df,alpha):
  def compute(row):
    if row['label'] == row['y_pred']:
      return row['weights'] * np.exp(-alpha)
    else:
      return row['weights'] * np.exp(alpha)
  df['updated_weights'] = df.apply(compute,axis=1)
  return df


# Step 6 : Normalize weights

In [23]:
def normalize_weights(df):
  total = df['updated_weights'].sum()
  df['normalized_weights'] = df['updated_weights'] / total
  return df

# **Step 7 : Compute cumulative bounds for sampling**

In [36]:
def compute_cumulative_bounds(df):
  df['cumsum_upper'] = np.cumsum(df['normalized_weights'])
  df['cumsum_lower'] = df['cumsum_upper'] - df['normalized_weights']
  return df

# **Step 8: Create new dataset based on normalized weights**

In [57]:
def create_new_dataset(df):
  indices = []
  for _ in range(df.shape[0]):
    a = np.random.random()
    # Select rows where 'cumsum_upper' is greater than 'a'
    filtered_df = df[df['cumsum_upper'] > a]

    # Check if the filtered DataFrame is empty
    if filtered_df.empty:
        # If empty, handle the case (e.g., select a random row)
        selected_row = df.sample(1).iloc[0]
    else:
        # If not empty, select the first row as before
        selected_row = filtered_df.iloc[0]

    indices.append(selected_row.name)
  return df.iloc[indices][['X1', 'X2', 'label', 'weights']]

# **Visualize dataset**

In [58]:
def plot_data(df):
  sns.scatterplot(x = df['X1'],y=df['X2'],hue=df['label'])
  plt.show()

# **Plot Decision Boundary**

In [59]:
def plot_decision_boundary(model,df):
  X = df[['X1','X2']].values
  y = df['label'].values
  plot_decision_regions(X,y,clf=model,legend=2)
  plt.show()

# **Visualize Decision Tree**

In [60]:
def show_tree(model):
  plot_tree(model)
  plt.show()

In [61]:
def run_adaboost(df, T=5):
    models = []
    alphas = []

    for t in range(T):
        print(f"\nBoosting round {t+1}")

        # Train model
        X = df[['X1', 'X2']].values
        y = df['label'].values
        model = train_decision_tree(X, y)
        df = add_predictions(df, model)

        # Calculate error
        error = (df['label'] != df['y_pred']).dot(df['weights'])
        if error == 0:
            alpha = 1e10  # Very large weight for perfect classifier
        elif error == 1:
            break  # Skip model with error 1 (worse than random)
        else:
            alpha = calculate_model_weight(error)

        print(f"Error: {error:.4f}, Alpha: {alpha:.4f}")
        models.append(model)
        alphas.append(alpha)

        # Update weights and resample
        df = update_weights(df, alpha)
        df = normalize_weights(df)
        df = compute_cumulative_bounds(df)
        df = create_new_dataset(df)

    return models, alphas


In [62]:
def ensemble_predict(models, alphas, X):
    final_score = np.zeros(X.shape[0])
    for model, alpha in zip(models, alphas):
        preds = model.predict(X)
        preds[preds == 0] = -1  # Convert to -1 for proper boosting sign
        final_score += alpha * preds
    return (final_score > 0).astype(int)


In [78]:
from sklearn.metrics import accuracy_score

# Compare predicted labels to true labels
accuracy = accuracy_score(df['label'], df['ensemble_pred'])
print(f"Ensemble Accuracy: {accuracy:.2f}")


Ensemble Accuracy: 0.50


In [79]:
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(df['label'], df['ensemble_pred'])
print("Confusion Matrix:")
print(conf_matrix)


Confusion Matrix:
[[0 5]
 [0 5]]


In [77]:
df = initialize_data()
models, alphas = run_adaboost(df, T=5)

# Predict using ensemble
X_test = df[['X1', 'X2']].values
ensemble_preds = ensemble_predict(models, alphas, X_test)

# Visualize ensemble performance
df['ensemble_pred'] = ensemble_preds
print(df[['label', 'ensemble_pred']])



Boosting round 1
Error: 0.3000, Alpha: 0.4660

Boosting round 2
Error: 0.2000, Alpha: 0.7625

Boosting round 3
Error: 0.0000, Alpha: 10000000000.0000

Boosting round 4
Error: 0.0000, Alpha: 10000000000.0000

Boosting round 5
Error: 0.0000, Alpha: 10000000000.0000
   label  ensemble_pred
0      1              1
1      1              1
2      0              1
3      1              1
4      0              1
5      1              1
6      0              1
7      1              1
8      0              1
9      0              1


# **Using Sklearn AdaBoostingClassifier**

In [64]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [65]:
X , y = make_classification(n_samples=100,n_features=2,n_informative=2,n_redundant=0,random_state=42)

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [72]:
# Weak Learner : Decision Stump
base_estimator = DecisionTreeClassifier(max_depth=1)

In [73]:
# AdaBoost Model
model = AdaBoostClassifier(estimator=base_estimator,n_estimators=50,learning_rate=1.0)

In [74]:
model.fit(X_train,y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1))

In [75]:
y_pred = model.predict(X_test)


In [76]:
print('Accuracy : ',accuracy_score(y_test,y_pred))

Accuracy :  0.95
